In [1]:
import importlib
import sys
from pathlib import Path

current = Path.cwd()

PROJECT_ROOT = None

for path in [current] + list(current.parents):
    if (path / "src").is_dir():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find project root containing 'src'.\n"
        f"Current working directory: {current}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

import src.datasets.echo_dataset as echo_dataset

echo_dataset = importlib.reload(echo_dataset)
EchoNetDataset = echo_dataset.EchoNetDataset

from src.models.echo.r2plus1d import R2Plus1DRegressor
import src.evaluation.evaluator as evaluator

evaluator = importlib.reload(evaluator)
collect_echo_predictions = evaluator.collect_echo_predictions

from src.evaluation.metrics import (
    regression_metrics,
    ef_threshold_metrics,
)
from src.utils.seed import set_seed

print("Imports successful.")

Project root: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service
Imports successful.


In [2]:
DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = DATA_DIR / "processed"
ECHO_PROCESSED_DIR = PROCESSED_DIR / "echo" / "echonet"

CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "echo"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "metrics"
    / "echo"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

set_seed(42)

print("Device:", DEVICE)
print("Processed Echo directory:", ECHO_PROCESSED_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)

Device: cuda
Processed Echo directory: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\echo\echonet
Checkpoint directory: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\checkpoints\echo


In [3]:
manifest_candidates = [
    ECHO_PROCESSED_DIR / "echonet_manifest.csv",
    ECHO_PROCESSED_DIR / "manifest.csv",
    ECHO_PROCESSED_DIR / "echonet.csv",
]

MANIFEST_PATH = None

for path in manifest_candidates:
    if path.exists():
        MANIFEST_PATH = path
        break

if MANIFEST_PATH is None:
    discovered = list(
        ECHO_PROCESSED_DIR.glob("*.csv")
    )

    if discovered:
        MANIFEST_PATH = discovered[0]

if MANIFEST_PATH is None:
    raise FileNotFoundError(
        "Could not find EchoNet manifest CSV."
    )

print("Manifest:", MANIFEST_PATH)

Manifest: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\echo\echonet\echonet_manifest.csv


In [4]:
manifest = pd.read_csv(MANIFEST_PATH)

print("Shape:", manifest.shape)
print()
print(manifest.head())
print()
print(manifest.columns.tolist())

Shape: (1530, 12)

           video_name                                         video_path  \
0  0X100009310A3BD7FC  data\raw\echo\echonet\Videos\0X100009310A3BD7F...   
1  0X1002E8FBACD08477  data\raw\echo\echonet\Videos\0X1002E8FBACD0847...   
2  0X1005D03EED19C65B  data\raw\echo\echonet\Videos\0X1005D03EED19C65...   
3  0X10075961BC11C88E  data\raw\echo\echonet\Videos\0X10075961BC11C88...   
4  0X10094BA0A028EAC3  data\raw\echo\echonet\Videos\0X10094BA0A028EAC...   

          EF  split  frame_count   fps  width  height  \
0  78.498406    val          174  50.0    112     112   
1  59.101988  train          215  50.0    112     112   
2  62.363798  train          104  50.0    112     112   
3  54.545097  train          122  55.0    112     112   
4  24.887742    val          207  52.0    112     112   

                                       frame_indices  lv_dysfunction  \
0  [23, 27, 31, 35, 39, 43, 47, 51, 55, 59, 63, 6...               0   
1  [43, 47, 51, 55, 59, 63, 67, 71, 7

In [5]:
required_columns = [
    "video_path",
    "EF",
]

missing_columns = [
    col
    for col in required_columns
    if col not in manifest.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

print("Required columns found.")

Required columns found.


In [6]:
if "split" in manifest.columns:
    print(
        manifest["split"]
        .value_counts(dropna=False)
    )

    test_manifest = manifest[
        manifest["split"]
        .astype(str)
        .str.lower()
        == "test"
    ].copy()

else:
    test_manifest = manifest.copy()

test_manifest = test_manifest.reset_index(
    drop=True
)

print("Test samples:", len(test_manifest))

split
train    1144
val       204
test      182
Name: count, dtype: int64
Test samples: 182


In [7]:
test_manifest["video_path"] = (
    test_manifest["video_path"]
    .astype(str)
)


def resolve_video_path(value):
    path = Path(value)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path.resolve()


resolved_paths = test_manifest["video_path"].map(
    resolve_video_path
)

exists_mask = resolved_paths.apply(
    Path.exists
)

print("Existing videos:", int(exists_mask.sum()))
print("Missing videos:", int((~exists_mask).sum()))

if not exists_mask.all():
    print(
        test_manifest.loc[
            ~exists_mask,
            ["video_path"]
        ].head(10)
    )

test_manifest = test_manifest[exists_mask].copy()
test_manifest["video_path"] = resolved_paths[exists_mask].astype(str)
test_manifest = test_manifest.reset_index(drop=True)

print(
    "Usable test videos:",
    len(test_manifest)
)

Existing videos: 182
Missing videos: 0
Usable test videos: 182


In [8]:
test_dataset = EchoNetDataset(
    manifest=test_manifest,
    split="test",
    num_frames=32,
    frame_stride=4,
    image_size=112,
    project_root=PROJECT_ROOT,
)

print("Dataset size:", len(test_dataset))

Dataset size: 182


In [9]:
sample = test_dataset[0]

print("Video shape:", sample["video"].shape)
print("EF:", sample["ef"].item())
print("Path:", sample["video_path"])

Video shape: torch.Size([3, 32, 112, 112])
EF: 55.95178985595703
Path: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\raw\echo\echonet\Videos\0X100CF05D141FF143.avi


In [10]:
TEST_BATCH_SIZE = 4

test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Batches:", len(test_loader))

Batches: 46


In [11]:
model = R2Plus1DRegressor(
    pretrained=False
).to(DEVICE)

print(
    "Model created on:",
    DEVICE
)

Model created on: cuda


In [12]:
checkpoint_candidates = [
    CHECKPOINT_DIR / "echonet_r2plus1d18.pt",
    CHECKPOINT_DIR / "best_model.pt",
    CHECKPOINT_DIR / "echonet_best.pt",
    CHECKPOINT_DIR / "r2plus1d_best.pt",
    CHECKPOINT_DIR / "model.pt",
]

CHECKPOINT_PATH = None

for path in checkpoint_candidates:
    if path.exists():
        CHECKPOINT_PATH = path
        break

if CHECKPOINT_PATH is None:
    discovered = list(
        CHECKPOINT_DIR.glob("*.pt")
    )

    if discovered:
        CHECKPOINT_PATH = discovered[0]

if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        f"No checkpoint found in {CHECKPOINT_DIR}"
    )

print("Checkpoint:", CHECKPOINT_PATH)

Checkpoint: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\checkpoints\echo\echonet_r2plus1d18.pt


In [13]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
)

if isinstance(checkpoint, dict):
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    else:
        state_dict = checkpoint

else:
    state_dict = checkpoint

clean_state_dict = {}

for key, value in state_dict.items():
    new_key = key

    if new_key.startswith("module."):
        new_key = new_key[len("module."):]

    clean_state_dict[new_key] = value

missing_keys, unexpected_keys = model.load_state_dict(
    clean_state_dict,
    strict=False,
)

print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

if len(missing_keys) == 0 and len(unexpected_keys) == 0:
    print("Checkpoint loaded successfully.")
else:
    print(
        "Checkpoint loaded with key differences."
    )

Missing keys: []
Unexpected keys: []
Checkpoint loaded successfully.


In [14]:
predictions = collect_echo_predictions(
    model=model,
    loader=test_loader,
    device=DEVICE,
)

y_true = predictions["y_true"]
y_pred = predictions["y_pred"]

print("Ground truth shape:", y_true.shape)
print("Prediction shape:", y_pred.shape)

Ground truth shape: (182,)
Prediction shape: (182,)


In [15]:
print("True EF range:")
print(
    float(np.min(y_true)),
    "to",
    float(np.max(y_true)),
)

print()

print("Predicted EF range:")
print(
    float(np.min(y_pred)),
    "to",
    float(np.max(y_pred)),
)

True EF range:
10.661108016967773 to 75.89429473876953

Predicted EF range:
25.00574493408203 to 62.29239273071289


In [16]:
metrics = regression_metrics(
    y_true,
    y_pred,
)

metrics

{'MAE': 7.522883897299295,
 'RMSE': 9.281847477604867,
 'R2': 0.40287300896153866}

In [17]:
threshold_metrics = ef_threshold_metrics(
    y_true,
    y_pred,
)

threshold_metrics

{'EF_<50_Accuracy': 0.8461538461538461, 'EF_<40_Accuracy': 0.9505494505494505}

In [18]:
all_metrics = {
    **metrics,
    **threshold_metrics,
}

results_df = pd.DataFrame(
    [all_metrics]
)

results_df

,MAE,RMSE,R2,EF_<50_Accuracy,EF_<40_Accuracy
0,7.522884,9.281847,0.402873,0.846154,0.950549


In [19]:
comparison = pd.DataFrame({
    "true_EF": y_true,
    "predicted_EF": y_pred,
})

comparison["absolute_error"] = (
    np.abs(
        comparison["true_EF"]
        - comparison["predicted_EF"]
    )
)

comparison.head(20)

,true_EF,predicted_EF,absolute_error
0,55.951790,53.756332,2.195457
1,41.014423,44.930676,3.916252
2,50.794720,47.748051,3.046669
3,50.634983,53.739273,3.104290
4,63.098087,54.661850,8.436237
5,58.494438,55.498386,2.996052
6,67.381569,58.301888,9.079681
7,57.284908,54.084133,3.200775
8,59.918606,57.700718,2.217888
9,61.026825,59.793015,1.233810


In [20]:
worst_predictions = comparison.sort_values(
    "absolute_error",
    ascending=False,
)

worst_predictions.head(20)

,true_EF,predicted_EF,absolute_error
137,61.392025,31.603556,29.788469
167,10.661108,36.269436,25.608328
128,27.628849,50.898834,23.269985
135,34.978882,54.539135,19.560253
30,68.286507,48.846626,19.439880
70,75.894295,56.854675,19.039619
42,50.160038,31.172161,18.987877
119,34.310337,53.172607,18.862270
87,68.913124,51.030682,17.882442
105,67.428337,49.826847,17.601490


In [21]:
metrics_path = (
    RESULTS_DIR
    / "echonet_test_metrics.csv"
)

results_df.to_csv(
    metrics_path,
    index=False,
)

print("Saved:", metrics_path)

Saved: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\metrics\echo\echonet_test_metrics.csv


In [22]:
predictions_df = pd.DataFrame({
    "video_path": predictions["video_paths"],
    "true_EF": y_true,
    "predicted_EF": y_pred,
})

predictions_df["absolute_error"] = (
    np.abs(
        predictions_df["true_EF"]
        - predictions_df["predicted_EF"]
    )
)

predictions_path = (
    RESULTS_DIR
    / "echonet_test_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False,
)

print("Saved:", predictions_path)

Saved: d:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\metrics\echo\echonet_test_predictions.csv


In [23]:
validation_manifest = manifest[
    manifest["split"].astype(str).str.lower() == "val"
].copy().reset_index(drop=True)

validation_paths = validation_manifest[
    "video_path"
].map(resolve_video_path)

validation_manifest = validation_manifest[
    validation_paths.map(Path.exists)
].copy().reset_index(drop=True)

validation_manifest["video_path"] = (
    validation_paths[
        validation_paths.map(Path.exists)
    ].astype(str).reset_index(drop=True)
)

validation_dataset = EchoNetDataset(
    manifest=validation_manifest,
    split="val",
    num_frames=32,
    frame_stride=4,
    image_size=112,
    project_root=PROJECT_ROOT,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
validation_predictions = collect_echo_predictions(
    model=model,
    loader=validation_loader,
    device=DEVICE,
)

pd.DataFrame({
    "video_path": validation_predictions["video_paths"],
    "true_EF": validation_predictions["y_true"],
    "predicted_EF": validation_predictions["y_pred"],
}).assign(
    absolute_error=lambda frame: np.abs(
        frame["true_EF"] - frame["predicted_EF"]
    )
).to_csv(
    RESULTS_DIR / "echonet_val_predictions.csv",
    index=False,
)
print("Saved real EchoNet validation predictions")

Saved real EchoNet validation predictions


In [24]:
print("EchoNet-Dynamic Test Evaluation")
print("-" * 40)
print(f"Test samples : {len(y_true)}")
print(f"MAE          : {metrics['MAE']:.4f}")
print(f"RMSE         : {metrics['RMSE']:.4f}")
print(f"R²           : {metrics['R2']:.4f}")
print(
    f"EF < 50 acc  : "
    f"{threshold_metrics['EF_<50_Accuracy']:.4f}"
)
print(
    f"EF < 40 acc  : "
    f"{threshold_metrics['EF_<40_Accuracy']:.4f}"
)

EchoNet-Dynamic Test Evaluation
----------------------------------------
Test samples : 182
MAE          : 7.5229
RMSE         : 9.2818
R²           : 0.4029
EF < 50 acc  : 0.8462
EF < 40 acc  : 0.9505
